In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/tts_project/vits

/content/drive/MyDrive/tts_project/vits


In [3]:
!pwd

/content/drive/MyDrive/tts_project/vits


In [4]:
!pip install gradio
!pip install deep-translator
!pip install phonemizer
!pip install unidecode
!pip install librosa
!pip install soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.0 MB/s eta 0:00:00


In [5]:
!ls configs

hindi_vits.json  ljs_base.json	ljs_nosdp.json	vctk_base.json


In [6]:
!ls logs/hindi_female_vits

config.json   D_96500.pth					   G_104000.pth
D_100000.pth  D_97000.pth					   G_104500.pth
D_100500.pth  D_97500.pth					   G_105000.pth
D_101000.pth  D_98000.pth					   G_105500.pth
D_101500.pth  D_98500.pth					   G_106000.pth
D_102000.pth  D_99000.pth					   G_106500.pth
D_102500.pth  D_99500.pth					   G_107000.pth
D_103000.pth  eval						   G_107500.pth
D_103500.pth  events.out.tfevents.1782292038.1b36b6a94c18.8408.0   G_108000.pth
D_104000.pth  events.out.tfevents.1782292981.58ada09bb582.2909.0   G_108500.pth
D_104500.pth  events.out.tfevents.1782300741.8cf46c6aeee7.7117.0   G_109000.pth
D_105000.pth  events.out.tfevents.1782309939.092af10cfb04.3640.0   G_109500.pth
D_105500.pth  events.out.tfevents.1782310693.092af10cfb04.7079.0   G_110000.pth
D_106000.pth  events.out.tfevents.1782322248.092af10cfb04.58455.0  G_110500.pth
D_106500.pth  events.out.tfevents.1782349744.b8353fc827a1.3749.0   G_111000.pth
D_107000.pth  events.out.tfevents.1782350066.d1ad4e0141d9.2183.0   G_111500.pt

In [7]:
import os
import torch
import gradio as gr
import soundfile as sf
from deep_translator import GoogleTranslator
from phonemizer import phonemize

import utils
import commons

from models import SynthesizerTrn
from text import text_to_sequence

In [8]:
config_path = "./configs/hindi_vits.json"

hps = utils.get_hparams_from_file(config_path)

print("✅ Config Loaded")
print(hps.data.sampling_rate)

✅ Config Loaded
22050


In [9]:
from text.symbols import symbols

hps.symbols = symbols

print("Symbols:", len(hps.symbols))

Symbols: 179


In [10]:
net_g = SynthesizerTrn(
    len(hps.symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model
)

net_g.eval()

print("✅ Model Created")

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


✅ Model Created


In [11]:
checkpoint_path = "./logs/hindi_female_vits/G_312200.pth"

In [12]:
!ls -lh ./logs/hindi_female_vits/

total 54G
-rw------- 1 root root 1.5K Jul 20 11:10 config.json
-rw------- 1 root root 536M Jul  7 12:58 D_100000.pth
-rw------- 1 root root 536M Jul  7 13:03 D_100500.pth
-rw------- 1 root root 536M Jul  7 13:08 D_101000.pth
-rw------- 1 root root 536M Jul  7 13:13 D_101500.pth
-rw------- 1 root root 536M Jul  7 13:18 D_102000.pth
-rw------- 1 root root 536M Jul  7 13:23 D_102500.pth
-rw------- 1 root root 536M Jul  7 13:28 D_103000.pth
-rw------- 1 root root 536M Jul  7 13:33 D_103500.pth
-rw------- 1 root root 536M Jul  7 13:38 D_104000.pth
-rw------- 1 root root 536M Jul  7 13:42 D_104500.pth
-rw------- 1 root root 536M Jul  7 13:47 D_105000.pth
-rw------- 1 root root 536M Jul  7 13:52 D_105500.pth
-rw------- 1 root root 536M Jul  7 13:57 D_106000.pth
-rw------- 1 root root 536M Jul  7 14:02 D_106500.pth
-rw------- 1 root root 536M Jul  7 14:07 D_107000.pth
-rw------- 1 root root 536M Jul  7 14:12 D_107500.pth
-rw------- 1 root root 536M Jul  7 14:17 D_108000.pth
-rw------- 1 root r

In [13]:
checkpoint_path = "./logs/hindi_female_vits/G_110000.pth"

In [14]:
import os

print(os.getcwd())

/content/drive/MyDrive/tts_project/vits


In [15]:
import os

print(os.path.exists("./logs/hindi_female_vits/G_110000.pth"))

True


In [17]:
!ls logs/hindi_female_vits/G_*.pth | sort -V | tail -1

logs/hindi_female_vits/G_118000.pth


In [18]:
utils.load_checkpoint(
    "./logs/hindi_female_vits/G_118000.pth",
    net_g,
    None
)

(SynthesizerTrn(
   (enc_p): TextEncoder(
     (emb): Embedding(179, 192)
     (encoder): Encoder(
       (drop): Dropout(p=0.1, inplace=False)
       (attn_layers): ModuleList(
         (0-5): 6 x MultiHeadAttention(
           (conv_q): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
           (conv_k): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
           (conv_v): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
           (conv_o): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
           (drop): Dropout(p=0.1, inplace=False)
         )
       )
       (norm_layers_1): ModuleList(
         (0-5): 6 x LayerNorm()
       )
       (ffn_layers): ModuleList(
         (0-5): 6 x FFN(
           (conv_1): Conv1d(192, 768, kernel_size=(3,), stride=(1,))
           (conv_2): Conv1d(768, 192, kernel_size=(3,), stride=(1,))
           (drop): Dropout(p=0.1, inplace=False)
         )
       )
       (norm_layers_2): ModuleList(
         (0-5): 6 x LayerNorm()
       )
     )
     (proj)

In [19]:
text = "Hello my name is Arav."

translated = GoogleTranslator(
    source="auto",
    target="hi"
).translate(text)

print(translated)

हेलो मेरा नाम अरव है.


In [20]:
text = "Hello, how are you?"
translated = GoogleTranslator(source="auto", target="hi").translate(text)

print("Translated:", translated)

Translated: नमस्ते, आप कैसे हैं?


In [21]:
from deep_translator import GoogleTranslator

def translate_text(text, output_language):

    language_map = {
        "Hindi": "hi",
        "English": "en",
        "Marathi": "mr",
        "Kannada": "kn"
    }

    translated = GoogleTranslator(
        source="auto",
        target=language_map[output_language]
    ).translate(text)

    return translated

In [22]:
!pip install deep-translator

In [23]:
def text_to_ipa(text):

    ipa = phonemize(
        text,
        language="hi",
        backend="espeak",
        strip=True
    )

    return ipa

In [24]:
!apt-get update -qq
!apt-get install -y espeak-ng

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 164 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng-data amd64 1.50+dfsg-10ubuntu0.1 [3,956 kB]
Get:4 http://archive

In [25]:
print(text_to_ipa("मेरा नाम आरव है"))

meːɾaː naːm aːɾəʋ hɛː


In [26]:
from phonemizer import phonemize

In [27]:
print(
    phonemize(
        "मेरा नाम आरव है",
        language="hi",
        backend="espeak"
    )
)

meːɾaː naːm aːɾəʋ hɛː 


In [28]:
!pip install phonemizer

In [29]:
def ipa_to_sequence(ipa):

    sequence = text_to_sequence(
        ipa,
        hps.data.text_cleaners
    )

    if hps.data.add_blank:
        sequence = commons.intersperse(sequence, 0)

    return torch.LongTensor(sequence)

In [30]:
ipa = text_to_ipa("मेरा नाम आरव है")

seq = ipa_to_sequence(ipa)

print(seq)
print(seq.shape)

tensor([  0,  55,   0,  47,   0, 158,   0, 125,   0,  43,   0, 158,   0,  16,
          0,  56,   0,  43,   0, 158,   0,  55,   0,  16,   0,  43,   0, 158,
          0, 125,   0,  83,   0, 136,   0,  16,   0,  50,   0,  86,   0, 158,
          0])
torch.Size([43])


In [31]:
def tts_generate(sequence):

    with torch.no_grad():

        x = sequence.unsqueeze(0)

        x_lengths = torch.LongTensor([sequence.size(0)])

        audio = net_g.infer(
            x,
            x_lengths,
            noise_scale=0.667,
            length_scale=1.0,
            noise_scale_w=0.8
        )[0][0, 0].cpu().numpy()

    return audio

In [32]:
audio = tts_generate(seq)

print(type(audio))
print(audio.shape)
print(audio[:10])

<class 'numpy.ndarray'>
(40192,)
[ 2.5827647e-05 -4.4566685e-05 -3.2316166e-06  4.7043795e-06
 -1.0362306e-05  5.7146608e-06  3.6243873e-06  2.3739885e-06
 -6.7076871e-06 -1.6180749e-05]


In [33]:
sf.write(
    "output.wav",
    audio,
    hps.data.sampling_rate
)

In [34]:
def save_audio(audio, filename="output.wav"):
    sf.write(
        filename,
        audio,
        hps.data.sampling_rate
    )
    return filename

In [35]:
audio_path = save_audio(audio)

print(audio_path)

output.wav


In [36]:
from IPython.display import Audio

Audio(audio_path)

In [37]:
def generate(text, output_language):

    translated = translate_text(text, output_language)

    # Generate speech only for Hindi
    if output_language == "Hindi":

        ipa = text_to_ipa(translated)

        seq = ipa_to_sequence(ipa)

        audio = tts_generate(seq)

        path = save_audio(audio)

        return translated, path

    # Other languages: translation only
    return translated, None

In [40]:
with gr.Blocks(
    title="IndicVoice AI",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🎤 IndicVoice AI

    ### Multilingual Translation + Hindi TTS
    """)

    text = gr.Textbox(
        label="Input Text",
        lines=5,
        placeholder="Type here..."
    )

    output_language = gr.Dropdown(
        choices=[
            "Hindi",
            "English",
            "Marathi",
            "Kannada"
        ],
        value="Hindi",
        label="Output Language"
    )

    generate_btn = gr.Button("🎙 Translate / Generate")

    translated = gr.Textbox(
        label="Translated Text"
    )

    audio = gr.Audio(
        label="Generated Speech (Hindi only)"
    )

    generate_btn.click(
        fn=generate,
        inputs=[
            text,
            output_language
        ],
        outputs=[
            translated,
            audio
        ]
    )

demo.launch()

/tmp/ipykernel_738/1525580994.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f68c5e77e6b37d2879.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/tts_project/vits

/content/drive/MyDrive/tts_project/vits


In [ ]:
!pwd
!ls logs/hindi_female_vits | tail

/content/drive/MyDrive/tts_project/vits
G_96000.pth
G_96500.pth
G_97000.pth
G_97500.pth
G_98000.pth
G_98500.pth
G_99000.pth
G_99500.pth
githash
train.log


In [ ]:
!ls -lh logs/hindi_female_vits | grep "G_" | sort -V | tail -5

-rw------- 1 root root 417M Jul  7 14:17 G_108000.pth
-rw------- 1 root root 417M Jul  7 14:22 G_108500.pth
-rw------- 1 root root 417M Jul  7 14:27 G_109000.pth
-rw------- 1 root root 417M Jul  7 14:32 G_109500.pth
-rw------- 1 root root 417M Jul  7 14:37 G_110000.pth


In [ ]:
!ls -lh logs/hindi_female_vits | grep "D_" | sort -V | tail -5

-rw------- 1 root root 536M Jul  7 14:17 D_108000.pth
-rw------- 1 root root 536M Jul  7 14:22 D_108500.pth
-rw------- 1 root root 536M Jul  7 14:27 D_109000.pth
-rw------- 1 root root 536M Jul  7 14:32 D_109500.pth
-rw------- 1 root root 536M Jul  7 14:37 D_110000.pth


In [ ]:
!pip install -q \
torch torchvision torchaudio \
numpy scipy pandas matplotlib \
librosa soundfile \
unidecode phonemizer inflect \
tensorboard tqdm \
cython numba \
g2p-en nltk \
pypinyin jieba \
gradio deep-translator

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 12.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 48.4 MB/s eta 0:00:00


In [ ]:
!apt-get update -qq
!apt-get install -y espeak-ng ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 145 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng

In [ ]:
%cd /content/drive/MyDrive/tts_project/vits

!python train.py -c configs/hindi_vits.json -m hindi_female_vits

/content/drive/MyDrive/tts_project/vits
2026-07-20 11:10:36.489020: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO:hindi_female_vits:{'train': {'log_interval': 10, 'eval_interval': 500, 'seed': 1234, 'epochs': 20000, 'learning_rate': 0.0002, 'betas': [0.8, 0.99], 'eps': 1e-09, 'batch_size': 4, 'fp16_run': True, 'lr_decay': 0.999875, 'segment_size': 4096, 'init_lr_ratio': 1, 'warmup_epochs': 0, 'c_mel': 45, 'c_kl': 1.0}, 'data': {'training_files': 'filelists/hindi_train_phoneme.txt', 'validation_files': 'filelists/hindi_val_phoneme.txt', 'text_cleaners': ['basic_cleaners'], 'max_wav_value': 32768.0, 'sampling_rate': 22050, 'filter_length': 1024, 'hop_length': 256, 'win_length': 1024, 'n_mel_channels': 80, 'mel_fmin': 0.0, 'mel_fmax': None, '